# `mlfinlab` Simple Example

This notebook follows the legacy `mlfinlab Release Hudson & Thames.pdf` notes for the filter examples in sections `4.2.1` and `4.2.2`.

It keeps the workflow intentionally small:

1. Load the packaged ETF price sample from `mlfinlab.datasets`.
2. Run the documented CUSUM and Z-score filters from `mlfinlab.filters`.
3. Visualize the sampled event times.
4. Inspect simple forward-return summaries for those events.

If you have already installed the package with `pip install -e RVUtils/mlfinlab`, the bootstrap cell below is a no-op. Otherwise it adds the vendored source tree directly to `sys.path` so the example can still run from inside ARBS.

In [ ]:
from pathlib import Path
import sys

package_root = Path("../../RVUtils/mlfinlab").resolve()
if str(package_root) not in sys.path:
    sys.path.insert(0, str(package_root))

import matplotlib.pyplot as plt
import pandas as pd

from mlfinlab.datasets import load_stock_prices
from mlfinlab.filters import cusum_filter, z_score_filter

plt.style.use("ggplot")

In [ ]:
prices = load_stock_prices()
close = prices["SPY"].dropna().rename("SPY")

summary = pd.DataFrame(
    {
        "start": [close.index.min()],
        "end": [close.index.max()],
        "observations": [close.shape[0]],
        "first_close": [close.iloc[0]],
        "last_close": [close.iloc[-1]],
    }
)
summary

In [ ]:
cusum_events = cusum_filter(close, threshold=2.0)
z_score_events = z_score_filter(close, mean_window=20, std_window=20, z_score=2.0)

event_counts = pd.Series(
    {
        "cusum_events": len(cusum_events),
        "z_score_events": len(z_score_events),
    }
).to_frame("count")
event_counts

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(close.index, close.values, label="SPY close", linewidth=1.5)
ax.scatter(cusum_events, close.loc[cusum_events], label="CUSUM events", color="tab:blue", s=18)
ax.scatter(z_score_events, close.loc[z_score_events], label="Z-score events", color="tab:orange", s=18)
ax.set_title("Legacy mlfinlab filters on packaged SPY sample")
ax.set_ylabel("Price")
ax.legend()
fig.autofmt_xdate()
plt.show()

In [ ]:
def forward_return_frame(close_series: pd.Series, events: pd.DatetimeIndex, horizon: int = 5) -> pd.DataFrame:
    frame = pd.DataFrame(index=pd.Index(events, name="event_time"))
    frame["close"] = close_series.reindex(frame.index)
    frame[f"forward_{horizon}d_return"] = close_series.shift(-horizon).reindex(frame.index) / frame["close"] - 1.0
    return frame.dropna()

cusum_forward = forward_return_frame(close, cusum_events)
z_score_forward = forward_return_frame(close, z_score_events)

pd.DataFrame(
    {
        "mean_forward_5d_return": [cusum_forward.iloc[:, 1].mean(), z_score_forward.iloc[:, 1].mean()],
        "median_forward_5d_return": [cusum_forward.iloc[:, 1].median(), z_score_forward.iloc[:, 1].median()],
        "sample_size": [len(cusum_forward), len(z_score_forward)],
    },
    index=["CUSUM", "Z-score"],
)